# Exp06 — IdiomBERT-v2 8-cell SCL/HNR/LI ablation (REVISION_PLAN Exp06)

Biggest remaining GPU commitment (~8 cells x ~100 min T4/A100). Toggles 3 MWE-specific training innovations on the QA-style Joint baseline (`Train_Join.py`'s architecture): **SCL** (span contrastive loss), **HNR** (hard-negative reweighting), **LI** (lateral inhibition at decode). 8 cells = every subset of {SCL, HNR, LI}.

**This is the only path to Main-tier for IdiomBERT** per the 2026-06-19 council verdict — everything else on the rigor queue (E3, E4, N1/N2) is polish, not a tier-mover.

**Two pre-existing bugs were fixed 2026-06-22 before this notebook could even be written** (caught by the repo's colab-preflight checklist):
1. `Train_Join` import path was stale from the 2026-06-19 reorg — `ModuleNotFoundError`.
2. Saved predictions used token indices instead of `Train_Join.py`'s char-offset schema — would NOT have registered through `Full_evaluation.py` silently.

Both verified via a real local smoke test (1 epoch, English-only, baseline cell) before this notebook was written.

**Persistence:** all outputs are symlinked to Drive by the runner; it hard-fails before training if a dir is not Drive-backed. If the session times out, **just re-run the training cell** — finished cells are skipped, it resumes at the first incomplete one.

Run cells top to bottom. Use a **GPU** runtime (Runtime → Change runtime type → T4/A100, A100 strongly preferred given the 8x cell count).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
CELLS    = 'baseline scl hnr li scl_hnr scl_li hnr_li scl_hnr_li'   # set to e.g. 'baseline' to run one cell
FORCE    = '0'                              # '1' retrains even completed cells
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
print('repo:', REPO, '| cells:', CELLS, '| force:', FORCE)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys
from pathlib import Path
# guard against the /content/Idiomator_Research/Idiomator_Research nesting bug
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)   # ABSOLUTE — the runner self-cds to git toplevel from here
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert Path('experiments/rigor/run_06b_idiombert_v2.sh').exists(), 'runner missing — wrong repo/branch? did you push run_06b?'

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive + export env the runner reads
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
os.environ['DRIVE_OUT'] = DRIVE_OUT
os.environ['CELLS']     = CELLS
os.environ['FORCE']     = FORCE
print('DRIVE_OUT =', DRIVE_OUT)

In [ ]:
# 5. GPU smoke test (~1-2 min): 1 epoch, English only, throwaway dir, baseline
#    cell — catches runtime/CUDA issues before the long run. Trains nothing kept.
#    (Already verified locally on CPU/MPS before this notebook was written —
#    this cell is the GPU-specific re-check, same convention as E3/E4's notebooks.)
!python experiments/rigor/run_06_idiombert_v2.py \
    --output_dir /tmp/_e06_smoke --langs English --test_langs English \
    --epochs 1 --batch_size 8

In [ ]:
# 6. Dry-run the real runner: prints the plan + persistence gate, trains nothing
!bash experiments/rigor/run_06b_idiombert_v2.sh --dry-run

In [ ]:
# 7. FULL EXP06 RUN — 8 cells, Drive-gated, resumable. THIS IS THE LONG ONE
#    (~8 x ~100 min). Re-run this cell after any timeout: finished cells skip,
#    resumes the rest. console.log streams to Drive (tee -a) so a partial log
#    survives a kill. Consider running CELLS=baseline first alone as a real-data
#    sanity check (baseline should land close to System E's Joint F1 — same
#    architecture/loss-weights, no toggles), before committing to all 8.
LOG = f'{DRIVE_OUT}/exp06_idiombertv2_console.log'
!bash experiments/rigor/run_06b_idiombert_v2.sh 2>&1 | tee -a "$LOG"
print('\nlog →', LOG)

In [ ]:
# 8. Persistence readback + 8-cell summary — reads metrics.json back FROM DRIVE
#    (not /content) so this confirms the results actually persisted.
import json
ALL_CELLS = 'baseline scl hnr li scl_hnr scl_li hnr_li scl_hnr_li'.split()
print(f"{'cell':<14}{'cls_f1':<10}{'span_em':<10}{'span_f1':<10}{'best_dev_f1':<14}{'persisted?'}")
rows = []
for cell in ALL_CELLS:
    mp = Path(DRIVE_OUT) / 'idiombert_v2' / cell / 'metrics.json'
    if not mp.exists():
        print(f'{cell:<14}— metrics.json NOT on Drive (incomplete / not persisted / not requested)')
        continue
    M = json.load(open(mp)); rows.append((cell, M))
    print(f"{cell:<14}{M['test_cls_macro_f1']:<10.4f}{M['test_span_em']:<10.4f}{M['test_span_f1']:<10.4f}{M['best_dev_f1']:<14.4f}yes")
if rows:
    print('\nNext: register each completed cell (Full_evaluation --joint_preds, cmds printed by the')
    print('runner), then compare Joint F1 across all 8 cells — baseline should land near System E\'s')
    print('Joint F1 as a sanity check (same encoder/loss-weights, no toggles); the other 7 cells are')
    print('the actual SCL/HNR/LI ablation matrix for the v2 preprint.')